# NN-VVC Phase F-3 — GPU Training on Google Colab

**Research context**: This notebook orchestrates the training of the Learned Image Compression (LIC) model and Intra Human Adapter (IHA) described in the NN-VVC research paper using OpenImages on CUDA GPU (Tesla T4 / Colab class).

## Storage Architecture

| Storage | Location | Persistent? |
|---|---|---|
| Colab local disk | `/content/` | ❌ Cleared on session end |
| Google Drive | `/content/drive/MyDrive/NN_VVC/` | ✅ Permanent storage |

**Key Guidelines**:
- All checkpoints and logs are written directly to Google Drive.
- Dataset archive `openimages_10k.zip` is extracted to local disk `/content/data` for fast SSD I/O.
- Safe default batch size: `--batch-size 4` with `--num-workers 2` on Tesla T4 (~14.5 GB VRAM).
- Proxy loss (Mask R-CNN ResNet-50 FPN) is active for research training.

## 15-Cell Workflow
1. GPU Detection
2. Mount Google Drive
3. Install Dependencies
4. Clone / Update Repository
5. Dataset Preparation
6. Dataset Manifest Verification
7. Cache Proxy Model Weights
8. CPU Smoke Test
9. GPU Smoke Test
10. 1-Epoch Full-Data Validation Test
11. Full LIC Research Training (Gated)
12. Resume Training
13. List Checkpoints
14. Evaluation
15. IHA Training

## Cell 1 — GPU Detection

In [ ]:
import torch
import subprocess

print('=' * 65)
print('NN-VVC System & GPU Diagnostic')
print('=' * 65)
print(f'CUDA available : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU Name       : {gpu_name}')
    print(f'GPU Memory     : {gpu_mem_gb:.2f} GB')
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=driver_version,cuda_version',
                                 '--format=csv,noheader'], capture_output=True, text=True)
        print(f'Driver / CUDA  : {result.stdout.strip()}')
    except Exception:
        pass
else:
    print('WARNING: No CUDA GPU detected. Training will run on CPU.')
    print('Go to: Runtime > Change runtime type > T4 GPU')

print(f'PyTorch version: {torch.__version__}')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT       = Path('/content/drive/MyDrive/NN_VVC')
REPO_DIR         = Path('/content/NN_VVC')
LOCAL_DATA_DIR   = Path('/content/data/openimages')

DRIVE_DATASET_ZIP = DRIVE_ROOT / 'openimages_10k.zip'
DRIVE_CKPT_LIC    = DRIVE_ROOT / 'checkpoints' / 'lic'
DRIVE_CKPT_IHA    = DRIVE_ROOT / 'checkpoints' / 'iha'
DRIVE_LOGS        = DRIVE_ROOT / 'logs'

for d in [DRIVE_ROOT, DRIVE_CKPT_LIC, DRIVE_CKPT_IHA, DRIVE_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

print('Google Drive connected.')
print(f'  Drive root      : {DRIVE_ROOT}')
print(f'  Checkpoint (LIC): {DRIVE_CKPT_LIC}')
print(f'  Logs            : {DRIVE_LOGS}')
print(f'  Dataset zip     : {DRIVE_DATASET_ZIP}')

## Cell 3 — Install Dependencies

In [ ]:
import subprocess, sys

extra_packages = ['scikit-image', 'pyyaml', 'tqdm', 'scipy', 'opencv-python']
for pkg in extra_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import torch, torchvision
print(f'Dependencies ready. PyTorch={torch.__version__}, torchvision={torchvision.__version__}')

## Cell 4 — Clone / Update Repository

In [ ]:
import subprocess, os, sys
from pathlib import Path

REPO_DIR = Path('/content/NN_VVC')
REPO_URL = 'https://github.com/Lucky-Gautam15/NN-VVC.git'

if REPO_DIR.exists():
    print('Updating repository...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    print('Cloning repository...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    
os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'Repository active at {os.getcwd()}')

## Cell 5 — Dataset Preparation (Unpack to Local SSD)

In [ ]:
from pathlib import Path
import subprocess

DRIVE_DATASET_ZIP = Path('/content/drive/MyDrive/NN_VVC/openimages_10k.zip')
LOCAL_DATA_DIR    = Path('/content/data/openimages')
TRAIN_DIR = LOCAL_DATA_DIR / 'train'
VAL_DIR   = LOCAL_DATA_DIR / 'val'

train_count = len(list(TRAIN_DIR.glob('*.png'))) if TRAIN_DIR.exists() else 0
val_count   = len(list(VAL_DIR.glob('*.png')))   if VAL_DIR.exists()   else 0

if train_count == 9000 and val_count == 1000:
    print(f'Dataset already extracted: {train_count} train, {val_count} val images.')
else:
    print(f'Extracting dataset from {DRIVE_DATASET_ZIP} ...')
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['python', 'scripts/package_f3_dataset.py', 'unpack',
         '--archive', str(DRIVE_DATASET_ZIP),
         '--output-dir', str(LOCAL_DATA_DIR)],
        check=True
    )
    print(f'Extracted: {len(list(TRAIN_DIR.glob("*.png")))} train, {len(list(VAL_DIR.glob("*.png")))} val images.')

## Cell 6 — Dataset Manifest Verification

In [ ]:
import json
from pathlib import Path

LOCAL_DATA_DIR = Path('/content/data/openimages')
MANIFEST_PATH  = LOCAL_DATA_DIR / 'manifest.json'

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    images = manifest.get('images', [])
    print(f'Manifest verified: {len(images)} entries recorded.')
else:
    print('Manifest not bundled in zip, checking image counts directly...')

t_imgs = len(list((LOCAL_DATA_DIR / 'train').rglob('*.png')))
v_imgs = len(list((LOCAL_DATA_DIR / 'val').rglob('*.png')))
print(f'Dataset verification: {t_imgs} train images, {v_imgs} val images (Total: {t_imgs + v_imgs})')
assert t_imgs == 9000 and v_imgs == 1000, f'Unexpected split: {t_imgs} train, {v_imgs} val'

## Cell 7 — Cache Proxy Model Weights to Google Drive

In [ ]:
import os
from pathlib import Path

# Set TORCH_HOME to Google Drive to persist Mask R-CNN weights (~170 MB)
DRIVE_TORCH_CACHE = Path('/content/drive/MyDrive/NN_VVC/torch_cache')
DRIVE_TORCH_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(DRIVE_TORCH_CACHE)
print(f'TORCH_HOME set to: {DRIVE_TORCH_CACHE}')

print('Pre-loading proxy model weights...')
from src.losses.proxy_loss import ProxyFeatureExtractor
_ = ProxyFeatureExtractor()  # Downloads on first run, cached thereafter
print('Proxy model weights ready.')

## Cell 8 — CPU Smoke Test

In [ ]:
import subprocess, sys

print('Running CPU smoke test (3 epochs, 64 images, no proxy loss)...')
subprocess.run(
    [
        sys.executable, 'scripts/train_f3.py', 'lic',
        '--data-dir', '/content/data/openimages/train',
        '--val-dir',  '/content/data/openimages/val',
        '--checkpoint-dir', '/content/smoke_ckpt/lic',
        '--log-dir',  '/content/smoke_logs',
        '--epochs', '3',
        '--batch-size', '4',
        '--device', 'cpu',
        '--no-proxy',
        '--no-amp',
        '--seed', '42',
        '--num-workers', '0',
        '--run-name', 'cpu_smoke',
        '--smoke',
        '--smoke-samples', '64',
        '--smoke-epochs', '3',
    ],
    check=True,
)
print('CPU smoke test PASSED.')

## Cell 9 — GPU Smoke Test

In [ ]:
import subprocess, sys, torch

if not torch.cuda.is_available():
    print('SKIP: CUDA GPU not available.')
else:
    print('Running GPU smoke test with AMP & Proxy loss...')
    subprocess.run(
        [
            sys.executable, 'scripts/train_f3.py', 'lic',
            '--data-dir', '/content/data/openimages/train',
            '--val-dir',  '/content/data/openimages/val',
            '--checkpoint-dir', '/content/smoke_ckpt_gpu/lic',
            '--log-dir',  '/content/smoke_logs_gpu',
            '--epochs', '3',
            '--batch-size', '4',
            '--device', 'cuda',
            '--use-amp',
            '--seed', '42',
            '--num-workers', '2',
            '--run-name', 'gpu_smoke',
            '--smoke',
            '--smoke-samples', '64',
            '--smoke-epochs', '3',
        ],
        check=True,
    )
    print('GPU smoke test PASSED.')

## Cell 10 — 1-Epoch Full-Data Validation Test

Runs 1 full epoch over all 9000 train + 1000 val images on GPU with batch size 4 and proxy loss enabled to ensure memory safety.

In [ ]:
import subprocess, sys

print('Running 1-epoch full-data validation test (9000 train + 1000 val, batch_size=4)...')
subprocess.run(
    [
        sys.executable, 'scripts/train_f3.py', 'lic',
        '--data-dir', '/content/data/openimages/train',
        '--val-dir',  '/content/data/openimages/val',
        '--checkpoint-dir', '/content/drive/MyDrive/NN_VVC/checkpoints/lic_val1ep',
        '--log-dir',  '/content/drive/MyDrive/NN_VVC/logs',
        '--epochs', '1',
        '--batch-size', '4',
        '--device', 'cuda',
        '--use-amp',
        '--seed', '42',
        '--num-workers', '2',
        '--run-name', 'lic_val_1ep',
    ],
    check=True,
)
print('1-Epoch full-data validation test PASSED.')

## Cell 11 — Full LIC Research Training (Gated)

⚠️ **MANUAL TRAINING GATE**: Remove the `raise RuntimeError` line below only when you are ready to launch long-running research training.

In [ ]:
# Remove the gate exception below to start training:
raise RuntimeError('TRAINING GATE: Remove this raise line when ready to launch full training.')

import subprocess, sys

DRIVE_CKPT_LIC = '/content/drive/MyDrive/NN_VVC/checkpoints/lic'
DRIVE_LOGS     = '/content/drive/MyDrive/NN_VVC/logs'

# Note: Default schedule is 50 epochs (or 320 epochs for full paper replication)
subprocess.run(
    [
        sys.executable, 'scripts/train_f3.py', 'lic',
        '--data-dir', '/content/data/openimages/train',
        '--val-dir',  '/content/data/openimages/val',
        '--checkpoint-dir', DRIVE_CKPT_LIC,
        '--log-dir',  DRIVE_LOGS,
        '--epochs', '50',
        '--batch-size', '4',
        '--device', 'cuda',
        '--use-amp',
        '--seed', '42',
        '--num-workers', '2',
        '--max-grad-norm', '1.0',
        '--val-freq', '1',
        '--save-freq', '1',
        '--run-name', 'lic_research_50ep',
    ],
    check=True,
)

## Cell 12 — Resume Training

In [ ]:
import subprocess, sys
from pathlib import Path

DRIVE_CKPT_LIC = Path('/content/drive/MyDrive/NN_VVC/checkpoints/lic')
DRIVE_LOGS     = '/content/drive/MyDrive/NN_VVC/logs'

epoch_ckpts = sorted(
    DRIVE_CKPT_LIC.glob('lic_epoch_*.pt'),
    key=lambda p: int(p.stem.split('_')[-1])
)

if not epoch_ckpts:
    print('No checkpoints found on Drive — starting fresh.')
    resume_args = []
else:
    latest = epoch_ckpts[-1]
    print(f'Resuming from latest checkpoint: {latest}')
    resume_args = ['--resume-from', str(latest)]

# Remove the gate exception below when ready to resume:
raise RuntimeError('RESUME GATE: Remove this raise line when ready to resume training.')

subprocess.run(
    [
        sys.executable, 'scripts/train_f3.py', 'lic',
        '--data-dir', '/content/data/openimages/train',
        '--val-dir',  '/content/data/openimages/val',
        '--checkpoint-dir', str(DRIVE_CKPT_LIC),
        '--log-dir',  DRIVE_LOGS,
        '--epochs', '50',
        '--batch-size', '4',
        '--device', 'cuda',
        '--use-amp',
        '--seed', '42',
        '--num-workers', '2',
        '--run-name', 'lic_research_50ep',
    ] + resume_args,
    check=True,
)

## Cell 13 — List Checkpoints & Logs

In [ ]:
from pathlib import Path
import torch

DRIVE_CKPT_LIC = Path('/content/drive/MyDrive/NN_VVC/checkpoints/lic')
ckpts = sorted(DRIVE_CKPT_LIC.glob('*.pt'))
print(f'Found {len(ckpts)} checkpoint(s):\n')

for ckpt in ckpts:
    data = torch.load(ckpt, map_location='cpu', weights_only=False)
    ep = data.get('epoch', '?')
    step = data.get('step', '?')
    t_loss = data.get('train_loss')
    v_loss = data.get('val_loss')
    t_loss_str = f'train_loss={t_loss:.4f}' if t_loss is not None else ''
    v_loss_str = f'val_loss={v_loss:.4f}' if v_loss is not None else ''
    print(f'  {ckpt.name:35s} | epoch={ep:>3} | step={step:>6} | {t_loss_str} {v_loss_str}')

## Cell 14 — Evaluation & BD-Rate Analysis

In [ ]:
from src.evaluation.bd_rate import calculate_bd_rate, calculate_bd_psnr
from src.evaluation.metrics import calculate_psnr, calculate_ssim
print('Evaluation modules loaded.')

## Cell 15 — IHA Training (Per-QP Adaptation)

In [ ]:
import subprocess, sys
from pathlib import Path

DRIVE_CKPT_LIC = Path('/content/drive/MyDrive/NN_VVC/checkpoints/lic')
DRIVE_CKPT_IHA = Path('/content/drive/MyDrive/NN_VVC/checkpoints/iha')
DRIVE_LOGS     = '/content/drive/MyDrive/NN_VVC/logs'

# Select target QP (e.g. 32)
QP = 32
LIC_CKPT = DRIVE_CKPT_LIC / f'lic_epoch_50.pt'

if not LIC_CKPT.exists():
    print(f'LIC checkpoint not found: {LIC_CKPT}')
else:
    raise RuntimeError('IHA GATE: Remove this raise line when ready to train IHA.')
    subprocess.run(
        [
            sys.executable, 'scripts/train_f3.py', 'iha',
            '--data-dir', '/content/data/openimages/train',
            '--val-dir',  '/content/data/openimages/val',
            '--lic-checkpoint', str(LIC_CKPT),
            '--qp', str(QP),
            '--checkpoint-dir', str(DRIVE_CKPT_IHA),
            '--log-dir',  DRIVE_LOGS,
            '--epochs', '50',
            '--batch-size', '4',
            '--device', 'cuda',
            '--use-amp',
            '--seed', '42',
            '--num-workers', '2',
            '--run-name', f'iha_qp{QP}',
        ],
        check=True,
    )